# Morning class 27/08 — Extra practice 09 SOLUTIONS: defining functions   (L06)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Extra practice 09 — Defining functions. Run this once.
raw_names = ["  Ana Lima ", "BO SILVA", "cai tan  ", " Dee  Okafor"]
temps_c = [0, 22, 37, -5]
readings = [3.2, 8.8, 1.0, 5.5]

print(raw_names)
print(temps_c)

### Question 1

Two converters. -> `0 -> 32.0`, `22 -> 71.6`, `37 -> 98.6`, `-5 -> 23.0`; then **`21.999999999999996`** and `False`.

The round trip does not come back to 22. `22 * 9 / 5 + 32` is 71.6, which
cannot be stored exactly in binary, and converting back inherits the error.

So `f_to_c(c_to_f(22)) == 22` is `False` on a pair of formulas that are
algebraically exact inverses. This is worksheet 04 Q8 arriving through a
function: **never compare two computed floats with `==`.** Compare with a
tolerance, or round at the point of display.

Note both functions `return`. `c_to_f(c_to_f(...))` composes only because
there is a value to feed back in.

In [ ]:
def c_to_f(c):
    return c * 9 / 5 + 32

def f_to_c(f):
    return (f - 32) * 5 / 9

for c in temps_c:
    print(c, "->", c_to_f(c))

print(f_to_c(c_to_f(22)))
print(f_to_c(c_to_f(22)) == 22)

### Question 2

Print versus return, again. -> `37.0` from `get_total(readings) * 2`, then `18.5` printed by `show_total`.

`get_total(readings) * 2` works because the call **evaluates to** 18.5.
`show_total(readings) * 2` would raise `TypeError: unsupported operand
type(s) for *: 'NoneType' and 'int'` — it prints, hands back `None`, and
`None * 2` is meaningless.

That is the practical test for worksheet 09 Q2: **can you use the call
inside a bigger expression?** If not, the function is a dead end — you can
see its answer but you cannot use it.

In [ ]:
def show_total(values):
    print(sum(values))

def get_total(values):
    return sum(values)

print(get_total(readings) * 2)

show_total(readings)
# show_total(readings) * 2 would raise TypeError: unsupported operand type(s)
# for *: 'NoneType' and 'int'. It printed, then handed back None, and None
# cannot be multiplied. A printing function is a dead end.

### Question 3

Splitting names. -> `Ana / Lima`, `BO / SILVA`, `cai / tan`, `Dee / Okafor`.

`.strip()` removes the surrounding whitespace and `.split()` with no
argument splits on **any run** of whitespace — which is why `' Dee  Okafor'`
with its double space still gives two parts. `split(" ")` would have given
an empty string in the middle.

`parts[-1]` rather than `parts[1]` so a middle name does not break it. It
still breaks on a single-word name and on `"van der Berg"`, which is the
usual fate of name-splitting code — worth a line in the docstring saying so.

In [ ]:
def split_name(full):
    parts = full.strip().split()
    return parts[0], parts[-1]

for raw in raw_names:
    first, last = split_name(raw)
    print(repr(raw), "->", first, "/", last)

### Question 4

Composition. -> `'Ana Lima' A.L.`, `'Bo Silva' B.S.`, `'Cai Tan' C.T.`, `'Dee  Okafor' D.O.`

`initials` never strips or lowercases anything — it calls `tidy` and works
on the result. One definition of "clean", used twice, so fixing it fixes
both.

`.strip().lower().title()` chains because each returns a **new string**.
Strings are immutable; nothing here modified `raw_names`.

Look at `'Dee  Okafor'` — `.title()` kept the double space, because `strip`
only touches the ends. `initials` still gets it right, because `.split()`
ignores runs of whitespace. Two functions, two different tolerances for
the same mess.

In [ ]:
def tidy(text):
    return text.strip().lower().title()

def initials(text):
    clean = tidy(text)
    letters = [word[0] for word in clean.split()]
    return ".".join(letters) + "."

for raw in raw_names:
    print(repr(raw), "->", repr(tidy(raw)), initials(raw))

### Question 5

A predicate. -> `0 -- freezing`, `22`/`37` above, `-5 -- freezing`, then `2 freezing`.

A function returning `True`/`False` reads naturally in an `if` and can be
dropped straight into a comprehension — the same object serving two
purposes.

The name matters as much as the code. `is_freezing(c)` says what the answer
means; `check(c)` would not. Predicates conventionally start with `is_`,
`has_` or `can_`.

And `0` counts as freezing here because the test is `<=`. That is a
decision, and the docstring is where it belongs.

In [ ]:
def is_freezing(c):
    """True if `c` degrees Celsius is at or below freezing."""
    return c <= 0

for c in temps_c:
    if is_freezing(c):
        print(c, "-- freezing")
    else:
        print(c, "-- above freezing")

print(len([c for c in temps_c if is_freezing(c)]), "freezing")

### Question 6

A summary dictionary. -> `{'count': 4, 'total': 18.5, 'mean': 4.625, 'smallest': 1.0, 'largest': 8.8}`, then `4.625`, then the docstring.

Returning a dict means the caller writes `stats(x)["mean"]` rather than
remembering that the mean is the third of five values. Adding a sixth
figure later breaks nobody.

The docstring records the assumption — non-empty — which is the thing that
will actually go wrong. `stats([])` raises `ZeroDivisionError` from the
mean, and the docstring is the only warning anyone gets.

It calls `stats(readings)` twice, computing everything a second time to
print one number. On four readings that is free; on four million it is not.

In [ ]:
def stats(values):
    """Summarise `values` as a dict. Assumes `values` is not empty."""
    return {
        "count": len(values),
        "total": sum(values),
        "mean": sum(values) / len(values),
        "smallest": min(values),
        "largest": max(values),
    }

print(stats(readings))
print(stats(readings)["mean"])
print(stats.__doc__)

### Question 7

Functions in a dictionary. -> `212.0`; `to_f -> 212.0`, `to_c -> 37.77777777777778`; then `<function c_to_f at 0x…>`.

`converters["to_f"]` gets the function **object**, and the `(100)` after it
calls it. Two separate steps that happen to be written next to each other.

This is worksheet 09 Q5 being useful: because a function is a value, it can
live in a dict, and the caller can pick behaviour by name at runtime — from
a config file, a command-line flag, a user choice. Python has no `switch`
statement and this is the usual replacement.

Put parentheses in the dictionary literal and you store the **result**
instead: `{"to_f": c_to_f(100)}` is `{"to_f": 212.0}`, computed once, and
not callable at all.

In [ ]:
converters = {"to_f": c_to_f, "to_c": f_to_c}

print(converters["to_f"](100))

for name, func in converters.items():
    print(name, "->", func(100))

print(converters["to_f"])

### Question 8

Absent versus falsy. -> `3.2`, `None`, `0`; then `is None:  False` and `not     : True`.

`safe_first([0])` returned a real value that happens to be falsy, and the
two tests disagree about it.

- `result is None` asks **was there an answer**. Correctly `False`.
- `not result` asks **is the answer empty-ish**, and `0`, `0.0`, `""` and
  `[]` all qualify. `True`, and wrong for the question being asked.

**Test for absence with `is None`.** This is the same distinction as
worksheet 14 Q1's careful handling of `"0.00"`, and it is the reason
`.get(key)` returning `None` is safer to check than to use.

In [ ]:
def safe_first(values):
    if not values:
        return None
    return values[0]

print(safe_first(readings))
print(safe_first([]))
print(safe_first([0]))

print("is None: ", safe_first([0]) is None)
print("not     :", not safe_first([0]))

# `safe_first([0])` returns 0 -- a real value that happens to be falsy. `is
# None` correctly says it is not missing; `not` says it is empty-ish, which
# is a different question. Test for absence with `is None`, never with `not`.

### Question 9

Calling before defining. -> `18.5`, then `NameError: name 'summarise_later' is not defined`.

`def` is not a declaration that the interpreter notices in advance — it is a
**statement that runs**, creating the name at the moment it executes.
Before that line runs, the name does not exist.

Inside another function this does not matter, because the body is not
executed until the call: a function can happily refer to something defined
later in the file, as long as it is there by the time anyone calls it
(worksheet 11 Q8).

At the top level of a notebook it matters a great deal, because **cell
order is execution order** and the two drift apart the moment you scroll up
to fix something. Restart & Run All is how you find out whether your
notebook still works from a clean kernel.

In [ ]:
print(get_total(readings))

print(summarise_later(readings))

def summarise_later(values):
    return sum(values) / len(values)

# This is SUPPOSED to raise: NameError: name 'summarise_later' is not
# defined.
#
# `def` is a statement that RUNS, creating the name when the interpreter
# reaches it. Until then the name does not exist. In a notebook that means
# cell order matters -- and re-running an earlier cell after deleting a
# later one is how a function quietly vanishes.